# Exploratory Data Analysis (EDA)

Here we will connect to the database and process the raw files to build the dataset before creating our model. To do that, we will: 
- Retrieve the datasets and join them
- Connection to a wheather API to add wheather information
- Perform analysis of categorical variables and codification
- Perform analysis of the numerical variables
- Perform outliers Detection
- Perform scaling

## 0. Set up and load libraries

In [1]:
import sqlalchemy as sa

import datetime
import requests
import meteostat as ms

from meteostat import Point
from requests.packages.urllib3.exceptions import InsecureRequestWarning

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

import plotly.express as px
import matplotlib.pyplot as plt
import missingno as msno
import seaborn as sns

******

## 1. Explore the database
Note: If we have the "TimeOout" error upon establishing connection, it could be cause when we connect to codespace a new IP is created and therefore, we need to update the entry rules in the security settings of our AWS rds database


In [ ]:
import os
from dotenv import load_dotenv

# Load variables from the .env file in the project root
load_dotenv()

USERNAME = os.environ["DB_USERNAME"]
PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = int(os.environ["DB_PORT"])
DB_NAME = os.environ["DB_NAME"]

LOCATION = (
    float(os.environ["SHOP_LAT"]),
    float(os.environ["SHOP_LON"]),
)

In [ ]:
import pymysql
import pandas as pd

class DatabaseConnection:
    def __init__(self, password, username=USERNAME,
                 host=DB_HOST,
                 port=DB_PORT,
                 database=DB_NAME):
        self.host = host
        self.port = port
        self.username = username
        self.password = password
        self.database = database

    # Function to execute a query and return a Pandas DataFrame
    def query_to_df(self, query):
        # Establish connection using pymysql directly
        connection = pymysql.connect(
            host=self.host,
            user=self.username,
            password=self.password,
            db=self.database,
            port=self.port
        )
        # Execute query and fetch data into a DataFrame
        try:
            df = pd.read_sql(query, connection)
        finally:
            connection.close()
        return df


In [ ]:
# Query to get all tables in the current database
query_show_tables = "SHOW TABLES"
database = DatabaseConnection(PASSWORD)

# Fetch the list of tables and print them
tables = database.query_to_df(query_show_tables)
print("Tables in the database:")
print(tables)


/tmp/ipykernel_24557/2994508414.py:6: DeprecationWarning: 'db' is deprecated, use 'database'
  tables = database.query_to_df(query_show_tables)


Tables in the database:
       Tables_in_panaderiasalvador
0                    articulos_top
1              calendario_completo
2                  calendario_dias
3                   raw_calendario
4                      raw_pedidos
5                       raw_ventas
6                   ventas_diarias
7           ventas_diarias_estudio
8  ventas_diarias_estudio_completo


/tmp/ipykernel_24557/915899454.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


Select the selling table named as "ventas" or "ventas_diarias" in the AWS database

In [ ]:
# Define the query
query_daily_and_top = """
SELECT familia,
articulo,
fechaVenta AS fecha_venta,
festivo,
precio,
cantidad,
importe,
orden_articulo_familia
FROM ventas_diarias_estudio
"""

# Instantiate the class with the password
database = DatabaseConnection(PASSWORD) 

# Execute the query and fetch results into a DataFrame
ventas = database.query_to_df(query_daily_and_top)

# Convert 'fecha_venta' to date format and 'articulo' to integer
ventas = ventas.assign(
    fecha_venta=pd.to_datetime(ventas["fecha_venta"])
).astype({"articulo": int})

# The 'ventas' DataFrame should now contain the query results
print(ventas.head())

/tmp/ipykernel_24557/1324976815.py:18: DeprecationWarning: 'db' is deprecated, use 'database'
  ventas = database.query_to_df(query_daily_and_top)


/tmp/ipykernel_24557/915899454.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


    familia  articulo fecha_venta festivo  precio  cantidad     importe  \
0  BOLLERIA      3960  2021-05-18     NaN   2.318     216.0  500.688008   
1  BOLLERIA      6286  2021-08-29     NaN   3.136      99.0  310.464005   
2  BOLLERIA      3880  2021-09-21     NaN   3.000     102.0  306.000000   
3  BOLLERIA      5803  2021-10-21     NaN   3.000      42.0  126.000000   
4  BOLLERIA      3880  2021-12-07     NaN   3.000     141.0  423.000000   

   orden_articulo_familia  
0                       1  
1                       4  
2                       3  
3                       5  
4                       3  


In [5]:
ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 10322 entries, 0 to 10321
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype        
---  ------                  --------------  -----        
 0   familia                 10322 non-null  str          
 1   articulo                10322 non-null  int64        
 2   fecha_venta             10322 non-null  datetime64[s]
 3   festivo                 809 non-null    str          
 4   precio                  10322 non-null  float64      
 5   cantidad                10322 non-null  float64      
 6   importe                 10322 non-null  float64      
 7   orden_articulo_familia  10322 non-null  int64        
dtypes: datetime64[s](1), float64(3), int64(2), str(2)
memory usage: 645.3 KB


In [6]:
ventas.head()

,familia,articulo,fecha_venta,festivo,precio,cantidad,importe,orden_articulo_familia
0,BOLLERIA,3960,2021-05-18,NaN,2.318,216.0,500.688008,1
1,BOLLERIA,6286,2021-08-29,NaN,3.136,99.0,310.464005,4
2,BOLLERIA,3880,2021-09-21,NaN,3.000,102.0,306.000000,3
3,BOLLERIA,5803,2021-10-21,NaN,3.000,42.0,126.000000,5
4,BOLLERIA,3880,2021-12-07,NaN,3.000,141.0,423.000000,3


In [7]:
ventas['festivo'].unique()

<StringArray>
[                                               nan,
                                  'Feria de Málaga',
                               'Noche de Halloween',
                                       'Nochevieja',
                                     'Martes Santo',
 'Día de la Hispanidad (Fiesta Nacional de España)',
                                     'Jueves Santo',
                          'Día de Todos los Santos',
                                          'Navidad',
                                  'Día de la Madre',
                                      'Lunes Santo',
                           'Día de la Constitución',
                                    'Día del Padre',
                                     'Día de Reyes',
                                 'Domingo de Ramos',
                                    'Viernes Santo',
                              'Día de San Valentín',
                                   'Día de la Cruz',
                          'Domin

In [8]:
ventas.sample(10)

,familia,articulo,fecha_venta,festivo,precio,cantidad,importe,orden_articulo_familia
9419,PANADERIA,1043,2021-06-30,NaN,2.596000,162.00,420.552000,2
802,PANADERIA,417,2022-12-12,NaN,4.615000,105.00,484.575000,5
5610,PASTELERIA,6549,2021-06-22,NaN,28.500000,3.00,85.500000,5
7150,BOLLERIA,5803,2021-11-10,NaN,3.000000,45.00,135.000000,5
6288,PASTELERIA,6451,2021-12-29,NaN,40.772999,15.00,611.595001,4
2715,BOLLERIA,6425,2022-06-21,NaN,34.091000,13.53,461.251305,2
3952,PANADERIA,1084,2021-10-01,NaN,0.865000,357.00,308.805008,4
8080,PANADERIA,900,2021-07-30,NaN,0.865000,483.00,417.794995,3
9355,PANADERIA,1043,2021-10-19,NaN,2.740000,177.00,484.980008,2
1033,PANADERIA,968,2022-11-24,NaN,2.452000,189.00,463.427999,1


******

## 2. Query to [meteostat API](https://dev.meteostat.net/python/daily.html)

Now we want to add the wheather data to the available datasets. 
Connecting to this app enables getting the following variables:

- **tavg** -> The average air temperature in °C
- **tmin** -> The minimum air temperature in °C
- **tmax** -> The maximum air temperature in °C
- **prcp** -> The daily precipitation total in mm
- **wdir** -> The average wind direction in degrees (°)
- **wspd** -> The average wind speed in km/h
- **pres** -> The average sea-level air pressure in hPa

### 2.1. Look for the location ID of the bakery shop

In [11]:
import datetime
import meteostat as ms

# Localization of the shop
point = ms.Point(36.721477644071705, -4.363132134392174)

# Find nearby weather stations
stations = ms.stations.nearby(point, limit=5)

print("Nearby stations:")
display(stations)

# Define when our selling data start and end

start = ventas["fecha_venta"].min()
end = ventas["fecha_venta"].max()

# Retrieve daily observations from those stations
ts = ms.daily(stations, start, end)
df = ts.fetch()
shop_location = stations.index[0]

if df is None or df.empty:
    print("No data found for these stations and dates.")
else:
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())
    print("The shop location ID is", shop_location)



Nearby stations:


,name,country,region,latitude,longitude,elevation,timezone,distance
id,,,,,,,,
08482,Malaga / Aeropuerto,ES,AN,36.6667,-4.4833,16,Europe/Madrid,12324.5


Shape: (730, 11)
Columns: ['temp', 'tmin', 'tmax', 'rhum', 'prcp', <Parameter.SNWD: 'snwd'>, 'wspd', 'wpgt', 'pres', <Parameter.TSUN: 'tsun'>, 'cldc']


temp  tmin  tmax  rhum  prcp  snwd  wspd  wpgt    pres  \
station time                                                                 
08482   2021-05-01  17.2  11.9  22.8    54   0.0  <NA>  14.2  <NA>  1014.0   
        2021-05-02  16.3  10.8  23.2    57   0.0  <NA>  15.7  <NA>  1015.9   
        2021-05-03  15.3  12.7  20.1    76   6.5  <NA>   9.7  <NA>  1017.7   
        2021-05-04  16.5  12.0  20.9    81   0.0  <NA>  12.1  <NA>  1018.0   
        2021-05-05  16.7  12.3  21.7    83   0.0  <NA>  13.8  <NA>  1016.5   

                    tsun  cldc  
station time                    
08482   2021-05-01  <NA>  <NA>  
        2021-05-02  <NA>  <NA>  
        2021-05-03  <NA>  <NA>  
        2021-05-04  <NA>  <NA>  
        2021-05-05  <NA>  <NA>

The shop location ID is 08482


### 2.2. Use shop location to create a function to retrieve the weather data

In [ ]:
import requests
import datetime
from meteostat import Point
from requests.packages.urllib3.exceptions import InsecureRequestWarning
class DailyWeatherData:

    # Localization of the shop
    SHOP_coords = shop_location

    # Columns identified as important
    IMP_COLUMNS = ['temp', 'tmin', 'tmax', 'prcp', 'wdir', 'wspd', 'pres']

    # Definition of the temporal horizon for the API query
    def __init__(self, start, end = datetime.datetime.now()):
        self.start = start
        self.end = end

    # Definition of the weather data retrieval method
    def get_weather_data(self):
        station_id = shop_location

        ts = ms.daily(station_id, self.start, self.end)
        data = ts.fetch()

        if data is None or data.empty:
            raise ValueError(
                f"No daily weather data available for station {station_id}"
            )

        available_columns = [
            col for col in self.IMP_COLUMNS if col in data.columns
        ]

        if not available_columns:
            raise ValueError(
                f"Expected weather columns not found. "
                f"Available columns: {data.columns.tolist()}"
            )

        return (
            data[available_columns]
            .add_suffix("_w")
            .reset_index()
        )


In [13]:
# Define when our selling data starts
# It can be defined in this format -> Example: datetime.datetime(2017, 1, 1)
start = ventas["fecha_venta"].min()

# Define when our selling data ends
# It can be defined in this format -> Example: datetime.datetime(2017, 1, 1)
end = ventas["fecha_venta"].max()

# Instantiate the class with the temporal horizon to query
weather_query = DailyWeatherData(start, end)

# Get the info of the weather data for the queried temporal horizon
weather = weather_query.get_weather_data()

weather.head()

,time,temp_w,tmin_w,tmax_w,prcp_w,wspd_w,pres_w
0,2021-05-01,17.2,11.9,22.8,0.0,14.2,1014.0
1,2021-05-02,16.3,10.8,23.2,0.0,15.7,1015.9
2,2021-05-03,15.3,12.7,20.1,6.5,9.7,1017.7
3,2021-05-04,16.5,12.0,20.9,0.0,12.1,1018.0
4,2021-05-05,16.7,12.3,21.7,0.0,13.8,1016.5


#### 2.3. Analyze weather data from the query

In [14]:
# Are there nulls in the data?
weather.isnull().sum()

time      0
temp_w    0
tmin_w    0
tmax_w    0
prcp_w    7
wspd_w    0
pres_w    0
dtype: int64

In [15]:
weather.describe()

,time,temp_w,tmin_w,tmax_w,prcp_w,wspd_w,pres_w
count,730,730.0,730.0,730.0,723.0,730.0,730.0
mean,2022-04-30 12:00:00,19.348493,15.174795,24.863836,0.817012,14.332055,1018.06
min,2021-05-01 00:00:00,7.3,1.1,13.3,0.0,6.1,1004.4
25%,2021-10-30 06:00:00,14.525,10.8,19.6,0.0,10.9,1014.5
50%,2022-04-30 12:00:00,19.25,14.6,24.65,0.0,12.5,1017.3
75%,2022-10-29 18:00:00,23.9,20.1,29.6,0.0,16.875,1021.0
max,2023-04-30 00:00:00,33.5,27.5,42.6,46.8,46.5,1032.8
std,NaN,5.402347,5.569168,5.984177,4.384574,5.323913,5.172823


In [16]:
weather.to_excel("/workspaces/Datathon_CodeSpace/Data/Datathon_weather_variables_20261020.xlsx", index = False)
weather.to_csv("/workspaces/Datathon_CodeSpace/Data/Datathon_weather_variables_20261020.csv", index = False)

*********************

## 3. Merge selling and whether data with join

In [19]:
# Join selling data and weather data using common datetime variable
df = pd.merge(ventas, weather,
              left_on=["fecha_venta"], right_on=["time"],
              how="inner").drop("time", axis=1)

# order the dataframe
df.sort_values(["fecha_venta", "articulo"], inplace = True)

# Convert data types if necessary
df['fecha_venta'] = pd.to_datetime(df['fecha_venta'], errors='coerce')

df.head(5)

,familia,articulo,fecha_venta,festivo,precio,cantidad,importe,orden_articulo_familia,temp_w,tmin_w,tmax_w,prcp_w,wspd_w,pres_w
8460,PANADERIA,417,2021-05-01,NaN,4.038,72.0,290.736004,5,17.2,11.9,22.8,0.0,14.2,1014.0
7412,PANADERIA,900,2021-05-01,NaN,0.865,651.0,563.114998,3,17.2,11.9,22.8,0.0,14.2,1014.0
4450,PANADERIA,968,2021-05-01,NaN,2.163,450.0,973.350002,1,17.2,11.9,22.8,0.0,14.2,1014.0
6173,PANADERIA,1043,2021-05-01,NaN,2.596,327.0,848.892002,2,17.2,11.9,22.8,0.0,14.2,1014.0
10262,PANADERIA,1084,2021-05-01,NaN,0.865,423.0,365.895005,4,17.2,11.9,22.8,0.0,14.2,1014.0


In [20]:
df.to_excel("/workspaces/Datathon_CodeSpace/Data/Datathon_processed_data_20261020.xlsx", index = False)
df.to_csv("/workspaces/Datathon_CodeSpace/Data/Datathon_processed_data_20261020.csv", index = False)

******

******